# SleepSense — Inference Demo (Google Colab)
**Team CC26-PSU230 | Coding Camp 2026 DBS Foundation**

Notebook ini mendemonstrasikan proses inference model SleepSense secara mandiri.
Jalankan sel secara berurutan dari atas ke bawah.

> ⚠️ Output adalah **SCREENING AWAL**, bukan diagnosis medis.

## Cell 1 — Upload Artifact Model

In [8]:
# Upload file artifact dari lokal:
#   - sleepsense_model.keras
#   - scaler_params.json
#   - feature_meta.json
# (atau langsung dari ZIP hasil training)

from google.colab import files
import os, zipfile

print("Upload sleepsense_flask_models.zip (hasil download dari notebook training)")
uploaded = files.upload()

# Ekstrak ZIP ke /content/models/
MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

for fname in uploaded:
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(MODELS_DIR)
        print(f"Extracted: {fname} -> {MODELS_DIR}/")
    else:
        # Jika upload file satu-satu
        os.rename(fname, os.path.join(MODELS_DIR, fname))
        print(f"Moved: {fname} -> {MODELS_DIR}/{fname}")

print("\nIsi folder models:")
for f in sorted(os.listdir(MODELS_DIR)):
    print(" ", f)

Upload sleepsense_flask_models.zip (hasil download dari notebook training)


Saving sleepsense_flask_hasil.zip to sleepsense_flask_hasil (1).zip
Extracted: sleepsense_flask_hasil (1).zip -> /content/models/

Isi folder models:
  feature_meta.json
  gradienttape_log.json
  scaler_params.json
  sleepsense_model.keras
  sleepsense_savedmodel
  tensorboard_logs
  test_metrics.json
  training_curves.png
  training_log.json


## Cell 2 — Import & Definisi Custom Components

In [9]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras          # <-- pakai tensorflow.keras, bukan import keras langsung
from tensorflow.keras import layers

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")

# ── Custom Layer ──────────────────────────────────────────────
class AttentionScaling(layers.Layer):
    """Custom attention layer — harus sama persis dengan saat training."""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.attention_dense = layers.Dense(units, activation="sigmoid", name="attn_gate")

    def call(self, inputs):
        return inputs * self.attention_dense(inputs)

    def get_config(self):
        cfg = super().get_config()
        cfg["units"] = self.units
        return cfg


# ── Custom Loss ───────────────────────────────────────────────
class FocalLoss(keras.losses.Loss):
    """Custom focal loss — harus sama persis dengan saat training."""
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        at     = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        return tf.reduce_mean(at * tf.pow(1.0 - p_t, self.gamma) * bce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma, "alpha": self.alpha})
        return cfg


print("Custom components siap.")

TensorFlow version : 2.20.0
Keras version      : 3.13.2
Custom components siap.


## Cell 3 — Load Model & Artifacts

In [10]:
MODELS_DIR  = "/content/models"
MODEL_PATH  = os.path.join(MODELS_DIR, "sleepsense_model.keras")
SCALER_PATH = os.path.join(MODELS_DIR, "scaler_params.json")
META_PATH   = os.path.join(MODELS_DIR, "feature_meta.json")

# Load scaler & metadata
with open(SCALER_PATH) as f:
    scaler_params = json.load(f)

with open(META_PATH) as f:
    meta = json.load(f)

# Load model dengan custom_objects
model = keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "AttentionScaling": AttentionScaling,
        "FocalLoss":        FocalLoss
    }
)

print(" Model berhasil dimuat.")
print(f"   Input shape  : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")
model.summary()

 Model berhasil dimuat.
   Input shape  : (None, 13)
   Output shape : (None, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'attention_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "SleepSense_StressRisk"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 13)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_1 (AttentionScaling)  │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stress_risk (Dense)             │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,813 (343.02 KB)

 Trainable params: 29,121 (113.75 KB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 58,244 (227.52 KB)

## Cell 4 — Fungsi Preprocessing & Interpretasi

In [11]:
import os  # sudah diimport di atas, tapi pastikan ada

def preprocess_input(user_input: dict) -> np.ndarray:
    """
    Mengubah input dictionary menjadi array siap inferens.
    Menerapkan StandardScaler (mean/scale dari training) dan
    one-hot encoding age group.
    """
    age       = float(user_input.get("age", 25))
    age_label = "teen" if age <= 18 else ("young_adult" if age <= 35 else "adult")

    row = {
        "age":                            age,
        "gender":                         float(meta["gender_map"].get(user_input.get("gender", "Male"), 0)),
        "sleep_duration_hours":           float(user_input.get("sleep_duration_hours", 7.0)),
        "sleep_quality_score":            float(user_input.get("sleep_quality_score", 5.0)),
        "daily_screen_time_hours":        float(user_input.get("daily_screen_time_hours", 4.0)),
        "pre_sleep_screen_time_hours":    float(user_input.get("pre_sleep_screen_time_hours", 1.0)),
        "physical_activity_minutes":      float(user_input.get("physical_activity_minutes", 30.0)),
        "caffeine_intake_cups":           float(user_input.get("caffeine_intake_cups", 2.0)),
        "mental_fatigue_score":           float(user_input.get("mental_fatigue_score", 5.0)),
        "notifications_received_per_day": float(user_input.get("notifications_received_per_day", 50.0)),
        "age_adult":       1.0 if age_label == "adult"       else 0.0,
        "age_teen":        1.0 if age_label == "teen"        else 0.0,
        "age_young_adult": 1.0 if age_label == "young_adult" else 0.0,
    }

    vec   = np.array([row[f] for f in scaler_params["feature_cols"]], dtype=np.float32)
    mean  = np.array(scaler_params["mean"],  dtype=np.float32)
    scale = np.array(scaler_params["scale"], dtype=np.float32)

    return ((vec - mean) / scale).reshape(1, -1)


def interpret(prob: float) -> dict:
    """Mengubah probabilitas model menjadi label risiko."""
    if prob < 0.35:
        level, label, emoji = "Rendah",  "No Risk",       "✅"
        summary = "Pola tidur dan gaya hidup Anda tergolong sehat."
    elif prob < 0.65:
        level, label, emoji = "Sedang",  "Moderate Risk", "⚠️"
        summary = "Ada beberapa aspek gaya hidup yang perlu diperhatikan."
    else:
        level, label, emoji = "Tinggi",  "At Risk",       "🔴"
        summary = "Pola tidur dan screen time Anda memerlukan perhatian segera."

    return {
        "risk_label":       label,
        "risk_level":       level,
        "risk_probability": round(float(prob), 4),
        "summary":          summary,
        "emoji":            emoji,
        "disclaimer":       "Ini adalah screening awal, BUKAN diagnosis medis."
    }


def predict(user_data: dict) -> dict:
    """Fungsi utama inference SleepSense."""
    x    = preprocess_input(user_data)
    prob = float(model(x, training=False).numpy().flatten()[0])
    return interpret(prob)


print("Fungsi inference siap.")

Fungsi inference siap.


## Cell 5 — Inference Demo: Satu Data Pengguna

In [12]:
# ── Ubah nilai di sini sesuai data yang ingin diprediksi ──────
sample_input = {
    "age":                            22,
    "gender":                         "Male",       # Male / Female / Other
    "sleep_duration_hours":           5.5,          # jam/malam
    "sleep_quality_score":            4.0,          # skala 1–10
    "daily_screen_time_hours":        8.0,          # jam/hari
    "pre_sleep_screen_time_hours":    2.5,          # jam sebelum tidur
    "physical_activity_minutes":      15,           # menit/hari
    "caffeine_intake_cups":           4,            # cangkir/hari
    "mental_fatigue_score":           7.5,          # skala 1–10
    "notifications_received_per_day": 120           # notifikasi/hari
}

result = predict(sample_input)

print("=" * 50)
print("         HASIL PREDIKSI SLEEPSENSE")
print("=" * 50)
print(f"  {result['emoji']}  Tingkat Risiko : {result['risk_level']} ({result['risk_label']})")
print(f"  Probabilitas  : {result['risk_probability']:.2%}")
print(f"  Keterangan    : {result['summary']}")
print(f"\n  {result['disclaimer']}")
print("=" * 50)

         HASIL PREDIKSI SLEEPSENSE
  ⚠️  Tingkat Risiko : Sedang (Moderate Risk)
  Probabilitas  : 63.85%
  Keterangan    : Ada beberapa aspek gaya hidup yang perlu diperhatikan.

  Ini adalah screening awal, BUKAN diagnosis medis.


## Cell 6 — Inference Batch: Beberapa Data Sekaligus

In [13]:
import pandas as pd

# Beberapa profil contoh dengan karakteristik berbeda
batch_samples = [
    {
        "label":                          "Profil Sehat",
        "age": 28, "gender": "Female",
        "sleep_duration_hours": 8.0,      "sleep_quality_score": 8.0,
        "daily_screen_time_hours": 3.0,   "pre_sleep_screen_time_hours": 0.5,
        "physical_activity_minutes": 60,  "caffeine_intake_cups": 1,
        "mental_fatigue_score": 3.0,      "notifications_received_per_day": 30
    },
    {
        "label":                          "Profil Risiko Sedang",
        "age": 30, "gender": "Male",
        "sleep_duration_hours": 6.5,      "sleep_quality_score": 5.5,
        "daily_screen_time_hours": 6.0,   "pre_sleep_screen_time_hours": 1.5,
        "physical_activity_minutes": 25,  "caffeine_intake_cups": 3,
        "mental_fatigue_score": 5.5,      "notifications_received_per_day": 80
    },
    {
        "label":                          "Profil Risiko Tinggi",
        "age": 22, "gender": "Male",
        "sleep_duration_hours": 5.0,      "sleep_quality_score": 3.0,
        "daily_screen_time_hours": 10.0,  "pre_sleep_screen_time_hours": 3.0,
        "physical_activity_minutes": 10,  "caffeine_intake_cups": 5,
        "mental_fatigue_score": 9.0,      "notifications_received_per_day": 150
    },
]

rows = []
for sample in batch_samples:
    label  = sample.pop("label")   # ambil label, sisanya input model
    result = predict(sample)
    rows.append({
        "Profil":          label,
        "Risiko":          result["risk_level"],
        "Label":           result["risk_label"],
        "Probabilitas":    f"{result['risk_probability']:.2%}",
        "Status":          result["emoji"],
    })

df_result = pd.DataFrame(rows)
print("Hasil Batch Inference:")
display(df_result)

Hasil Batch Inference:


,Profil,Risiko,Label,Probabilitas,Status
0,Profil Sehat,Rendah,No Risk,29.65%,✅
1,Profil Risiko Sedang,Sedang,Moderate Risk,50.22%,⚠️
2,Profil Risiko Tinggi,Tinggi,At Risk,68.41%,🔴
